# 4.05 Structured · HTML · Code Loaders

PDF/웹이 아닌 **구조화 데이터**(CSV·JSON)와 **경량 텍스트 포맷**(HTML·Markdown·Git 저장소)을 LangChain `Document`로 변환한다.
또한 IBM `docling`의 DocumentConverter를 붙인 `DoclingLoader`와 범용 `UnstructuredFileLoader`를 살펴본다.

## 학습 목표

- `CSVLoader`의 `source_column` · `metadata_columns` · `content_columns`로 메타데이터를 분리한다
- `JSONLoader`의 `jq_schema`와 `metadata_func`로 필드 단위 문서화를 한다
- `BSHTMLLoader` / `UnstructuredFileLoader`로 HTML·Markdown을 로드한다
- `GitLoader`로 공개 저장소를 얕은 clone 해 소스코드를 `Document`로 읽어온다
- `DoclingLoader`의 `ExportType`(chunks vs markdown) 차이를 본다

## 언제 쓰나

- 스프레드시트/제품 카탈로그/QA 쌍 — `CSVLoader`
- API dump, 로그, 구조화된 설정 파일 — `JSONLoader`
- 블로그·사내 위키 HTML export — `BSHTMLLoader`
- 코드베이스 RAG(README + 함수 시그니처) — `GitLoader` + 언어별 splitter

## 4.05.1 환경 설정

필요 패키지: `langchain-community`, `beautifulsoup4`, `jq`, `unstructured`, `langchain-docling`(선택), `docling`(선택). CSV/JSON/HTML 샘플 파일을 probe에서 직접 생성한다.

In [ ]:
# !pip install -U langchain-community beautifulsoup4 jq unstructured langchain-docling docling
import os, json, csv
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)

SAMPLE_DIR = Path("./_struct_samples").resolve()
SAMPLE_DIR.mkdir(exist_ok=True)

# 1) CSV — 고객 문의 데이터
CSV_PATH = SAMPLE_DIR / "faq.csv"
with CSV_PATH.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "topic", "question", "answer"])
    w.writerow(["f1", "billing", "환불은 며칠 걸리나요?", "영업일 기준 3~5일"])
    w.writerow(["f2", "account", "비밀번호를 잊었어요", "로그인 화면의 '찾기' 링크"])
    w.writerow(["f3", "shipping", "배송 추적은 어디서?", "주문 상세 페이지에서 확인"])

# 2) JSON — 상품 카탈로그(배열)
JSON_PATH = SAMPLE_DIR / "products.json"
JSON_PATH.write_text(json.dumps([
    {"sku": "SKU-1", "name": "맥북", "price": 1990000, "tags": ["laptop", "apple"]},
    {"sku": "SKU-2", "name": "키보드", "price": 89000, "tags": ["peripheral"]},
    {"sku": "SKU-3", "name": "모니터", "price": 450000, "tags": ["display"]},
], ensure_ascii=False, indent=2), encoding="utf-8")

# 3) HTML
HTML_PATH = SAMPLE_DIR / "handbook.html"
HTML_PATH.write_text("""<!doctype html>
<html><head><title>Handbook</title></head>
<body>
<h1>Engineering Handbook</h1>
<p>On-call rotates every Monday. Pagers are carried for 7 days.</p>
<h2>Deployments</h2>
<p>All deploys require a PR with a green CI and two reviewers.</p>
</body></html>""", encoding="utf-8")

# 4) Markdown (UnstructuredFileLoader 입력)
MD_PATH = SAMPLE_DIR / "notes.md"
MD_PATH.write_text("# Notes\n\n- item one\n- item two\n\n## Section\n\nbody text.\n", encoding="utf-8")

assert all(p.exists() for p in [CSV_PATH, JSON_PATH, HTML_PATH, MD_PATH])
print("samples:", [p.name for p in SAMPLE_DIR.iterdir()])

## 4.05.2 CSV — 메타데이터 분리와 content 선택

`CSVLoader`는 **한 행당 하나의 `Document`** 를 만든다. 기본 동작은 전체 컬럼을 `key: value` 형태로 펼쳐 `page_content`에 넣는 것이지만, 실무에서는

- `source_column`: 원본 식별자 컬럼을 `metadata['source']`로 빼내기
- `metadata_columns`: 검색 필터용 컬럼을 `metadata`로 빼내기
- `content_columns`: 실제 본문만 `page_content`에 남기기

를 조합해 **본문과 필터 키를 분리**하는 것이 RAG에 유리하다.

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    str(CSV_PATH),
    source_column="id",
    metadata_columns=("topic",),
    content_columns=("question", "answer"),
)
docs = loader.load()
for d in docs:
    print("meta:", {k: d.metadata[k] for k in ('source', 'topic')})
    print("text:", d.page_content)
    print("---")

## 4.05.3 JSON — `jq_schema`와 `metadata_func`

`JSONLoader`는 내부적으로 `jq` 라이브러리로 JSON을 순회한다.

- `jq_schema`: 어느 경로를 하나의 레코드로 볼지. 최상위 배열이면 `.[]`, 객체 배열의 특정 필드만 뽑으려면 `.[].name`.
- `content_key`: 각 레코드에서 어느 키를 `page_content`로 쓸지.
- `metadata_func(record, metadata) -> metadata`: 레코드의 나머지 필드를 `metadata`로 승격.

In [ ]:
from langchain_community.document_loaders import JSONLoader

def enrich(record: dict, metadata: dict) -> dict:
    metadata["sku"]   = record.get("sku")
    metadata["price"] = record.get("price")
    metadata["tags"]  = ",".join(record.get("tags", []))
    return metadata

loader = JSONLoader(
    file_path=str(JSON_PATH),
    jq_schema=".[]",
    content_key="name",
    metadata_func=enrich,
)
for d in loader.load():
    print(d.page_content, "|", {k: d.metadata[k] for k in ('sku', 'price', 'tags')})

## 4.05.4 HTML · Markdown · Unstructured

`BSHTMLLoader`(BeautifulSoup 기반)는 간단한 HTML을 빠르게 텍스트로 펼친다.
`UnstructuredFileLoader`는 파일 확장자를 보고 `unstructured`의 파서를 라우팅해 PDF·PPTX·DOCX·MD 등 **수십 종을 한 API로** 처리한다.

In [ ]:
from langchain_community.document_loaders import BSHTMLLoader, UnstructuredFileLoader

html_docs = BSHTMLLoader(str(HTML_PATH), get_text_separator="\n").load()
print("== BSHTMLLoader ==")
print("title:", html_docs[0].metadata.get("title"))
print(html_docs[0].page_content[:200])

md_docs = UnstructuredFileLoader(str(MD_PATH)).load()
print("\n== UnstructuredFileLoader (markdown) ==")
print(md_docs[0].page_content[:200])

## 4.05.5 GitLoader — 공개 저장소 얕은 clone

`GitLoader(repo_path, clone_url=..., branch=..., file_filter=...)`는 로컬에 없으면 clone 후 파일 하나씩 `Document`로 만든다.
`file_filter`로 확장자를 제한하고, 이후 `Language.PYTHON` 기반 `RecursiveCharacterTextSplitter.from_language`로 **함수 경계를 존중하는 청크**를 만들 수 있다.

아래 예시는 일부러 매우 작은 공개 저장소를 선택한다 — 실행 시간과 디스크 사용을 줄이기 위해.

In [ ]:
import tempfile
from langchain_community.document_loaders import GitLoader
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

tmp_repo = Path(tempfile.mkdtemp(prefix="git_loader_")) / "repo"

loader = GitLoader(
    repo_path=str(tmp_repo),
    clone_url="https://github.com/octocat/Hello-World.git",  # 수 KB의 데모 저장소
    branch="master",
    file_filter=lambda p: not p.endswith((".png", ".jpg", ".gif")),
)
files = loader.load()
print("파일 수:", len(files))
for d in files[:5]:
    print("-", d.metadata.get("file_path"), "|", len(d.page_content), "bytes")

# 언어 인식 splitter 예시 (실제 Python 파일이 있을 때)
py_splitter = RecursiveCharacterTextSplitter.from_language(Language.PYTHON, chunk_size=400, chunk_overlap=0)
sample = """def add(a, b):\n    return a + b\n\nclass Calc:\n    def mul(self, a, b):\n        return a * b\n"""
print("\nPython 청크 경계:", py_splitter.split_text(sample))

## 4.05.6 DoclingLoader — 레이아웃 보존 + 청크

IBM `docling`은 PDF/DOCX/PPTX/HTML 등을 **문서 구조(섹션·표·리스트)** 를 유지한 중간표현으로 변환한 뒤, LangChain 쪽 어댑터(`langchain-docling`)가 두 가지 export 모드를 제공한다.

- `ExportType.DOC_CHUNKS`(기본): 구조 기반 청크 리스트로 바로 split
- `ExportType.MARKDOWN`: 단일 markdown 문서로 export — 후속 splitter 자유도

아래 셀은 앞서 만든 HTML 파일을 입력으로 써서 경량으로 확인한다.

In [ ]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

chunked = DoclingLoader(file_path=str(HTML_PATH), export_type=ExportType.DOC_CHUNKS).load()
print("chunked docs:", len(chunked))
print("첫 청크 미리보기:", chunked[0].page_content[:150].replace("\n", " "))

md = DoclingLoader(file_path=str(HTML_PATH), export_type=ExportType.MARKDOWN).load()
print("\nmarkdown export:")
print(md[0].page_content[:200])

## 4.05.7 split · embed 연동 — 한 줄 RAG

CSV에서 만든 `Document`에 splitter를 돌려 `Chroma`로 밀어넣으면 FAQ 검색이 바로 완성된다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

assert os.environ.get("OPENAI_API_KEY"), "Chroma 임베딩용 OPENAI_API_KEY 필요"

faq_docs = CSVLoader(
    str(CSV_PATH), source_column="id", metadata_columns=("topic",),
    content_columns=("question", "answer"),
).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0).split_documents(faq_docs)

store = Chroma.from_documents(chunks, embedding=OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="faq_demo")
for hit in store.similarity_search("돈 돌려받는 기간", k=2):
    print("-", hit.metadata.get("source"), "|", hit.metadata.get("topic"), "|", hit.page_content)

## 체크리스트

- [ ] CSV는 `source_column` + `metadata_columns` + `content_columns`로 본문·필터 분리
- [ ] JSON은 `jq_schema`로 범위를 좁히고 `metadata_func`로 필드 승격
- [ ] HTML/MD 단일 파일은 `BSHTMLLoader`·`UnstructuredFileLoader`, 복잡한 문서는 `DoclingLoader`
- [ ] 코드베이스는 `GitLoader` + `RecursiveCharacterTextSplitter.from_language(Language.X)` 조합

## 다음

- `../05_retrievers/` — 로더 + 벡터스토어 위에 앉히는 retriever
- `../06_text_splitters/` — 언어 인식 splitter와 markdown-header splitter

## 참고

- CSV/JSON 가이드: https://python.langchain.com/docs/how_to/document_loader_csv/ · https://python.langchain.com/docs/how_to/document_loader_json/
- Docling: https://github.com/DS4SD/docling
- Unstructured: https://docs.unstructured.io/